# 02. 가설 검정 EDA

전처리된 데이터와 원본 데이터를 함께 사용해 발표에서 다룬 가설을 확인한 코드입니다.

- 주문 연도·요일·주말, 블랙프라이데이·추수감사절·크리스마스 연휴 효과
- 동일 국가/동일 주(State) 배송, Haversine 거리
- 할인율 구간 × 결제 방식 × 카테고리, 할인율 0% 주문(Cleats) 분석, TRANSFER 결제의 낮은 지연율

> 셀 출력은 저장되어 있지 않습니다. 결과 수치와 그래프는 [발표 자료](../docs/presentation.pdf) 10~16쪽을 참고하세요.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path("../data")
# processed_train_MS.csv: 팀 공통 전처리 결과 (최종 Baseline 모델 학습 데이터)


In [ ]:
# 전처리 후 전체 피처와 타깃의 상관계수 확인

processed_train = pd.read_csv(DATA_DIR / "processed_train_MS.csv")

target = "Late_delivery_risk"

target_corr = (
    processed_train
    .corr(numeric_only=True)[target]
    .drop(target)
    .sort_values(key=lambda x: x.abs(), ascending=False)
    .to_frame("correlation_with_target")
)

target_corr["abs_correlation"] = target_corr["correlation_with_target"].abs()

display(target_corr)


In [ ]:
# 전처리 후 데이터 불러오기

processed_train = pd.read_csv(DATA_DIR / "processed_train_MS.csv")

print("processed_train shape:", processed_train.shape)
processed_train.head()


In [ ]:
# 새 파생변수 포함 describe().T

desc_T = processed_train.describe().T

desc_T["missing"] = processed_train.isnull().sum()
desc_T["n_unique"] = processed_train.nunique()
desc_T["dtype"] = processed_train.dtypes

desc_T = desc_T[
    ["dtype", "missing", "n_unique", "count", "mean", "std", "min", "25%", "50%", "75%", "max"]
]

display(desc_T)


In [ ]:
# 새 파생변수 포함 전체 수치형 히스토그램: 타깃 0/1 비교

target = "Late_delivery_risk"

num_cols = processed_train.select_dtypes(include=np.number).columns.tolist()
num_cols = [col for col in num_cols if col != target]

n_cols = 4
n_rows = int(np.ceil(len(num_cols) / n_cols))

plt.figure(figsize=(5 * n_cols, 3.5 * n_rows))

for i, col in enumerate(num_cols, 1):
    plt.subplot(n_rows, n_cols, i)
    
    plt.hist(
        processed_train.loc[processed_train[target] == 0, col].dropna(),
        bins=40,
        alpha=0.5,
        label="0",
        color="steelblue"
    )
    
    plt.hist(
        processed_train.loc[processed_train[target] == 1, col].dropna(),
        bins=40,
        alpha=0.5,
        label="1",
        color="salmon"
    )
    
    plt.title(col)
    plt.xlabel("")
    plt.ylabel("count")
    plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# order_year가 2018인 row의 order_month 확인

processed_train = pd.read_csv(DATA_DIR / "processed_train_MS.csv")

rows_2018 = processed_train[processed_train["order_year"] == 2018]

print("2018 rows:", rows_2018.shape[0])

display(
    rows_2018["order_month"]
    .value_counts()
    .sort_index()
    .to_frame("count")
)


In [ ]:
# order_year가 2015인 row의 order_month 확인

processed_train = pd.read_csv(DATA_DIR / "processed_train_MS.csv")

rows_2018 = processed_train[processed_train["order_year"] == 2015]

print("20185rows:", rows_2018.shape[0])

display(
    rows_2018["order_month"]
    .value_counts()
    .sort_index()
    .to_frame("count")
)


In [ ]:
# 주중/주말별 Late_delivery_risk 0/1 비율 비교

target = "Late_delivery_risk"

processed_train = pd.read_csv(DATA_DIR / "processed_train_MS.csv")

processed_train["day_type"] = processed_train["is_weekend"].map({
    0: "weekday",
    1: "weekend"
})

count_table = pd.crosstab(
    processed_train["day_type"],
    processed_train[target]
)

ratio_table = pd.crosstab(
    processed_train["day_type"],
    processed_train[target],
    normalize="index"
)

weekend_summary = pd.concat(
    [
        count_table.add_prefix("count_"),
        ratio_table.add_prefix("ratio_")
    ],
    axis=1
)

display(weekend_summary)


In [ ]:
# 요일별 Late_delivery_risk 0/1 비율 비교

target = "Late_delivery_risk"

processed_train = pd.read_csv(DATA_DIR / "processed_train_MS.csv")

weekday_map = {
    0: "Mon",
    1: "Tue",
    2: "Wed",
    3: "Thu",
    4: "Fri",
    5: "Sat",
    6: "Sun"
}

processed_train["weekday_name"] = processed_train["order_weekday"].map(weekday_map)

count_table = pd.crosstab(
    processed_train["weekday_name"],
    processed_train[target]
).reindex(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])

ratio_table = pd.crosstab(
    processed_train["weekday_name"],
    processed_train[target],
    normalize="index"
).reindex(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])

weekday_summary = pd.concat(
    [
        count_table.add_prefix("count_"),
        ratio_table.add_prefix("ratio_")
    ],
    axis=1
)

display(weekday_summary)


In [ ]:
# 블랙프라이데이 여부 생성 후 Late_delivery_risk 비교
# 블랙프라이데이 = 11월 넷째 금요일

raw_train = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")
processed_train = pd.read_csv(DATA_DIR / "processed_train_MS.csv")

DATE_COL = "order date (DateOrders)"
target = "Late_delivery_risk"

dt = pd.to_datetime(raw_train[DATE_COL], errors="coerce")

bf_dates = {
    2015: pd.Timestamp("2015-11-27"),
    2016: pd.Timestamp("2016-11-25"),
    2017: pd.Timestamp("2017-11-24"),
    2018: pd.Timestamp("2018-11-23"),
}

black_friday_date = dt.dt.year.map(bf_dates)

processed_train["order_date"] = dt.dt.date
processed_train["is_black_friday"] = (dt.dt.normalize() == black_friday_date).astype(int)

print("Black Friday rows:", processed_train["is_black_friday"].sum())

bf_summary = pd.crosstab(
    processed_train["is_black_friday"],
    processed_train[target],
    normalize="index"
)

bf_count = pd.crosstab(
    processed_train["is_black_friday"],
    processed_train[target]
)

bf_result = pd.concat(
    [
        bf_count.add_prefix("count_"),
        bf_summary.add_prefix("ratio_")
    ],
    axis=1
)

display(bf_result)


In [ ]:
# 블랙프라이데이 주간: 블랙프라이데이 전후 3일

bf_date_series = black_friday_date
days_from_bf = (dt.dt.normalize() - bf_date_series).dt.days

processed_train["is_black_friday_week"] = days_from_bf.between(-3, 3).astype(int)

print("Black Friday week rows:", processed_train["is_black_friday_week"].sum())

bf_week_summary = pd.crosstab(
    processed_train["is_black_friday_week"],
    processed_train[target],
    normalize="index"
)

bf_week_count = pd.crosstab(
    processed_train["is_black_friday_week"],
    processed_train[target]
)

display(
    pd.concat(
        [
            bf_week_count.add_prefix("count_"),
            bf_week_summary.add_prefix("ratio_")
        ],
        axis=1
    )
)


In [ ]:
# 미국 주요 연휴 직전~당일 주문건 여부 확인

raw_train = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")
processed_train = pd.read_csv(DATA_DIR / "processed_train_MS.csv")

DATE_COL = "order date (DateOrders)"
target = "Late_delivery_risk"

dt = pd.to_datetime(raw_train[DATE_COL], errors="coerce").dt.normalize()

def thanksgiving_date(year):
    # 11월 넷째 목요일
    nov = pd.date_range(
        start=f"{year}-11-01",
        end=f"{year}-11-30",
        freq="D"
    )
    thursdays = nov[nov.weekday == 3]
    return thursdays[3]

years = sorted(dt.dt.year.dropna().unique().astype(int))

holiday_windows = {}

for year in years:
    tg = thanksgiving_date(year)
    bf = tg + pd.Timedelta(days=1)
    christmas = pd.Timestamp(f"{year}-12-25")
    new_year = pd.Timestamp(f"{year}-01-01")
    
    holiday_windows[f"thanksgiving_{year}"] = (tg - pd.Timedelta(days=7), tg)
    holiday_windows[f"black_friday_{year}"] = (bf - pd.Timedelta(days=7), bf)
    holiday_windows[f"christmas_{year}"] = (christmas - pd.Timedelta(days=14), christmas)
    holiday_windows[f"new_year_{year}"] = (new_year - pd.Timedelta(days=7), new_year)

processed_train["holiday_pre_window"] = 0
processed_train["holiday_name"] = "none"

for name, (start, end) in holiday_windows.items():
    mask = dt.between(start, end)
    processed_train.loc[mask, "holiday_pre_window"] = 1
    processed_train.loc[mask, "holiday_name"] = name.rsplit("_", 1)[0]

print("holiday_pre_window rows:", processed_train["holiday_pre_window"].sum())

holiday_summary = pd.crosstab(
    processed_train["holiday_pre_window"],
    processed_train[target],
    normalize="index"
)

holiday_count = pd.crosstab(
    processed_train["holiday_pre_window"],
    processed_train[target]
)

display(
    pd.concat(
        [
            holiday_count.add_prefix("count_"),
            holiday_summary.add_prefix("ratio_")
        ],
        axis=1
    )
)


In [ ]:
# 미국 주요 연휴: 당일 3일 전부터 당일까지 주문건 여부 확인

raw_train = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")
processed_train = pd.read_csv(DATA_DIR / "processed_train_MS.csv")

DATE_COL = "order date (DateOrders)"
target = "Late_delivery_risk"

dt = pd.to_datetime(raw_train[DATE_COL], errors="coerce").dt.normalize()

def thanksgiving_date(year):
    # 11월 넷째 목요일
    nov = pd.date_range(
        start=f"{year}-11-01",
        end=f"{year}-11-30",
        freq="D"
    )
    thursdays = nov[nov.weekday == 3]
    return thursdays[3]

years = sorted(dt.dt.year.dropna().unique().astype(int))

holiday_dates = {}

for year in years:
    thanksgiving = thanksgiving_date(year)
    black_friday = thanksgiving + pd.Timedelta(days=1)
    christmas = pd.Timestamp(f"{year}-12-25")
    new_year = pd.Timestamp(f"{year}-01-01")
    
    holiday_dates[f"thanksgiving_{year}"] = thanksgiving
    holiday_dates[f"black_friday_{year}"] = black_friday
    holiday_dates[f"christmas_{year}"] = christmas
    holiday_dates[f"new_year_{year}"] = new_year

processed_train["holiday_3days_before"] = 0
processed_train["holiday_name"] = "none"

for name, holiday_date in holiday_dates.items():
    start = holiday_date - pd.Timedelta(days=3)
    end = holiday_date
    
    mask = dt.between(start, end)
    
    processed_train.loc[mask, "holiday_3days_before"] = 1
    processed_train.loc[mask, "holiday_name"] = name.rsplit("_", 1)[0]

print("holiday_3days_before rows:", processed_train["holiday_3days_before"].sum())

holiday_summary = pd.crosstab(
    processed_train["holiday_3days_before"],
    processed_train[target],
    normalize="index"
)

holiday_count = pd.crosstab(
    processed_train["holiday_3days_before"],
    processed_train[target]
)

display(
    pd.concat(
        [
            holiday_count.add_prefix("count_"),
            holiday_summary.add_prefix("ratio_")
        ],
        axis=1
    )
)


In [ ]:
holiday_name_summary = (
    processed_train
    .groupby("holiday_name")[target]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
)

display(holiday_name_summary)


In [ ]:
# 연휴 종류별 Late_delivery_risk 0/1 비율

target = "Late_delivery_risk"

holiday_ratio = pd.crosstab(
    processed_train["holiday_name"],
    processed_train[target],
    normalize="index"
)

holiday_count = pd.crosstab(
    processed_train["holiday_name"],
    processed_train[target]
)

holiday_ratio_summary = pd.concat(
    [
        holiday_count.add_prefix("count_"),
        holiday_ratio.add_prefix("ratio_")
    ],
    axis=1
).sort_values("ratio_1", ascending=False)

display(holiday_ratio_summary)


In [ ]:
# Christmas: 당일 3일 전부터 당일까지 주문건의 Late_delivery_risk 0/1 비율

christmas_df = processed_train[processed_train["holiday_name"] == "christmas"]

print("christmas rows:", len(christmas_df))

christmas_count = christmas_df["Late_delivery_risk"].value_counts().sort_index()
christmas_ratio = christmas_df["Late_delivery_risk"].value_counts(normalize=True).sort_index()

christmas_summary = pd.DataFrame({
    "count": christmas_count,
    "ratio": christmas_ratio
})

display(christmas_summary)


In [ ]:
target = "Late_delivery_risk"
discount_col = "Order Item Discount Rate"

discount_summary = (
    processed_train
    .groupby(discount_col)[target]
    .agg(["count", "mean"])
    .reset_index()
    .sort_values(discount_col)
)

discount_summary["ratio_1"] = discount_summary["mean"]
discount_summary["ratio_0"] = 1 - discount_summary["mean"]

display(discount_summary)


In [ ]:
processed_train["discount_bin"] = pd.cut(
    processed_train[discount_col],
    bins=[-0.001, 0, 0.05, 0.10, 0.15, 0.20, 0.25],
    labels=["0%", "0-5%", "5-10%", "10-15%", "15-20%", "20-25%"]
)

discount_bin_count = pd.crosstab(
    processed_train["discount_bin"],
    processed_train[target]
)

discount_bin_ratio = pd.crosstab(
    processed_train["discount_bin"],
    processed_train[target],
    normalize="index"
)

discount_bin_summary = pd.concat(
    [
        discount_bin_count.add_prefix("count_"),
        discount_bin_ratio.add_prefix("ratio_")
    ],
    axis=1
)

display(discount_bin_summary)


In [ ]:
# Christmas window + 할인율 조합과 타깃 비율 분석
# Christmas window = 크리스마스 당일 3일 전부터 당일까지

raw_train = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")
processed_train = pd.read_csv(DATA_DIR / "processed_train_MS.csv")

DATE_COL = "order date (DateOrders)"
target = "Late_delivery_risk"
discount_col = "Order Item Discount Rate"

dt = pd.to_datetime(raw_train[DATE_COL], errors="coerce").dt.normalize()

christmas_dates = {
    year: pd.Timestamp(f"{year}-12-25")
    for year in sorted(dt.dt.year.dropna().unique().astype(int))
}

processed_train["is_christmas_window"] = 0

for year, christmas_date in christmas_dates.items():
    start = christmas_date - pd.Timedelta(days=3)
    end = christmas_date
    
    mask = dt.between(start, end)
    processed_train.loc[mask, "is_christmas_window"] = 1

print("Christmas window rows:", processed_train["is_christmas_window"].sum())


In [ ]:
# 할인율 구간 생성

processed_train["discount_bin"] = pd.cut(
    processed_train[discount_col],
    bins=[-0.001, 0, 0.05, 0.10, 0.15, 0.20, 0.25],
    labels=["0%", "0-5%", "5-10%", "10-15%", "15-20%", "20-25%"]
)

christmas_discount_summary = (
    processed_train
    .groupby(["is_christmas_window", "discount_bin"])[target]
    .agg(["count", "mean"])
    .reset_index()
)

christmas_discount_summary["period"] = christmas_discount_summary["is_christmas_window"].map({
    0: "non_christmas",
    1: "christmas"
})

display(christmas_discount_summary)


In [ ]:
# Black Friday window + 할인율 조합과 타깃 비율 분석
# Black Friday window = 블랙프라이데이 당일 3일 전부터 당일까지

raw_train = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")
processed_train = pd.read_csv(DATA_DIR / "processed_train_MS.csv")

DATE_COL = "order date (DateOrders)"
target = "Late_delivery_risk"
discount_col = "Order Item Discount Rate"

dt = pd.to_datetime(raw_train[DATE_COL], errors="coerce").dt.normalize()

def thanksgiving_date(year):
    # 11월 넷째 목요일
    nov = pd.date_range(
        start=f"{year}-11-01",
        end=f"{year}-11-30",
        freq="D"
    )
    thursdays = nov[nov.weekday == 3]
    return thursdays[3]

years = sorted(dt.dt.year.dropna().unique().astype(int))

processed_train["is_black_friday_window"] = 0

for year in years:
    thanksgiving = thanksgiving_date(year)
    black_friday = thanksgiving + pd.Timedelta(days=1)
    
    start = black_friday - pd.Timedelta(days=3)
    end = black_friday
    
    mask = dt.between(start, end)
    processed_train.loc[mask, "is_black_friday_window"] = 1

print("Black Friday window rows:", processed_train["is_black_friday_window"].sum())


In [ ]:
# 할인율 구간 생성

processed_train["discount_bin"] = pd.cut(
    processed_train[discount_col],
    bins=[-0.001, 0, 0.05, 0.10, 0.15, 0.20, 0.25],
    labels=["0%", "0-5%", "5-10%", "10-15%", "15-20%", "20-25%"]
)

black_friday_discount_summary = (
    processed_train
    .groupby(["is_black_friday_window", "discount_bin"])[target]
    .agg(["count", "mean"])
    .reset_index()
)

black_friday_discount_summary["period"] = black_friday_discount_summary["is_black_friday_window"].map({
    0: "non_black_friday",
    1: "black_friday"
})

display(black_friday_discount_summary)


In [ ]:
# same_country / same_state별 배송 지연률 비교

processed_train = pd.read_csv(DATA_DIR / "processed_train_MS.csv")

target = "Late_delivery_risk"

for col in ["same_country", "same_state"]:
    print(f"\n=== {col} ===")
    
    summary = (
        processed_train
        .groupby(col)[target]
        .agg(["count", "mean"])
        .rename(columns={"mean": "late_risk_rate"})
    )
    
    summary["ratio"] = summary["count"] / len(processed_train)
    
    display(summary)


In [ ]:
# same_state 여부별 count / risk rate
processed_train.groupby("same_state")["Late_delivery_risk"].agg(["count", "mean"])


In [ ]:
# 하버사인 거리(dist_cust_to_order_region)와 Late_delivery_risk 관계 한 번에 보기

dist_col = "dist_cust_to_order_region"
target = "Late_delivery_risk"

# 1. 기본 통계
print("=== Distance Describe ===")
display(processed_train[dist_col].describe().to_frame().T)

# 2. 거리 구간 생성
processed_train["distance_bin"] = pd.qcut(
    processed_train[dist_col],
    q=10,
    duplicates="drop"
)

# 3. 거리 구간별 지연률
distance_summary = (
    processed_train
    .groupby("distance_bin")[target]
    .agg(["count", "mean"])
    .rename(columns={"mean": "late_risk_rate"})
)

print("\n=== Distance Bin Risk Rate ===")
display(distance_summary)

# 4. 거리 구간 x same_state 지연률
distance_state_pivot = processed_train.pivot_table(
    index="distance_bin",
    columns="same_state",
    values=target,
    aggfunc="mean"
)

distance_state_count = processed_train.pivot_table(
    index="distance_bin",
    columns="same_state",
    values=target,
    aggfunc="count"
)

print("\n=== Distance Bin x same_state Risk Rate ===")
display(distance_state_pivot)

print("\n=== Distance Bin x same_state Count ===")
display(distance_state_count)

# 5. 시각화
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 거리 전체 분포
sns.histplot(
    processed_train[dist_col],
    bins=50,
    kde=True,
    ax=axes[0, 0]
)
axes[0, 0].set_title("Distance Distribution")
axes[0, 0].set_xlabel("Distance (km)")
axes[0, 0].set_ylabel("Count")

# 타깃별 거리 분포
sns.histplot(
    processed_train.loc[processed_train[target] == 0, dist_col],
    bins=50,
    alpha=0.5,
    label="0",
    color="steelblue",
    ax=axes[0, 1]
)

sns.histplot(
    processed_train.loc[processed_train[target] == 1, dist_col],
    bins=50,
    alpha=0.5,
    label="1",
    color="salmon",
    ax=axes[0, 1]
)

axes[0, 1].set_title("Distance Distribution by Target")
axes[0, 1].set_xlabel("Distance (km)")
axes[0, 1].set_ylabel("Count")
axes[0, 1].legend(title=target)

# 거리 구간별 지연률
sns.barplot(
    data=distance_summary.reset_index(),
    x="distance_bin",
    y="late_risk_rate",
    ax=axes[1, 0]
)
axes[1, 0].set_title("Late Delivery Risk Rate by Distance Bin")
axes[1, 0].set_xlabel("Distance Bin")
axes[1, 0].set_ylabel("Risk Rate")
axes[1, 0].tick_params(axis="x", rotation=45)

# 거리 구간 x same_state 히트맵
sns.heatmap(
    distance_state_pivot,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    ax=axes[1, 1]
)
axes[1, 1].set_title("Risk Rate: Distance Bin x same_state")
axes[1, 1].set_xlabel("same_state")
axes[1, 1].set_ylabel("Distance Bin")

plt.tight_layout()
plt.show()


In [ ]:
# 원본 Category Name 기준 Late_delivery_risk 비율

raw_train = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")

target = "Late_delivery_risk"
cat_col = "Category Name"

raw_category_summary = (
    raw_train
    .groupby(cat_col)[target]
    .agg(["count", "mean"])
    .rename(columns={"mean": "risk_rate"})
    .sort_values("risk_rate", ascending=False)
)

display(raw_category_summary)


In [ ]:
# 묶은 카테고리(category_group) 기준 Late_delivery_risk 0/1 비율
# 원본 Category Name에 CAT_GROUP_MAP 다시 적용해서 이름으로 보기

raw_train = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")

target = "Late_delivery_risk"

CAT_GROUP_MAP = {
    "Women's Apparel":"Apparel",
    "Girls' Apparel":"Apparel",
    "Women's Clothing":"Apparel",
    "Men's Clothing":"Apparel",
    "Children's Clothing":"Apparel",
    "Baby":"Apparel",
    "Cleats":"Footwear",
    "Men's Footwear":"Footwear",
    "Golf Shoes":"Footwear",
    "Cardio Equipment":"Fitness",
    "Strength Training":"Fitness",
    "Fitness Accessories":"Fitness",
    "Golf Balls":"Golf",
    "Golf Gloves":"Golf",
    "Golf Apparel":"Golf",
    "Men's Golf Clubs":"Golf",
    "Women's Golf Clubs":"Golf",
    "Kids' Golf Clubs":"Golf",
    "Golf Bags & Carts":"Golf",
    "Camping & Hiking":"Outdoor",
    "Fishing":"Outdoor",
    "Water Sports":"Outdoor",
    "Indoor/Outdoor Games":"Outdoor",
    "Hunting & Shooting":"Outdoor",
    "Baseball & Softball":"Sports",
    "Hockey":"Sports",
    "Boxing & MMA":"Sports",
    "Tennis & Racquet":"Sports",
    "Soccer":"Sports",
    "Basketball":"Sports",
    "Sporting Goods":"Sports",
    "Accessories":"Sports",
    "Electronics":"Electronics",
    "Cameras":"Electronics",
    "Computers":"Electronics",
    "Consumer Electronics":"Electronics",
    "Toys":"Toys_Games",
    "Video Games":"Toys_Games",
    "DVDs":"Media",
    "Music":"Media",
    "Books":"Media",
    "CDs":"Media",
    "Health and Beauty":"Health_Beauty",
    "Pet Supplies":"Pet",
    "Crafts":"Other"
}

raw_train["category_group_name"] = (
    raw_train["Category Name"]
    .map(CAT_GROUP_MAP)
    .fillna("Other")
)

category_group_count = pd.crosstab(
    raw_train["category_group_name"],
    raw_train[target]
)

category_group_ratio = pd.crosstab(
    raw_train["category_group_name"],
    raw_train[target],
    normalize="index"
)

category_group_summary = pd.concat(
    [
        category_group_count.add_prefix("count_"),
        category_group_ratio.add_prefix("ratio_")
    ],
    axis=1
).sort_values("ratio_1", ascending=False)

display(category_group_summary)


In [ ]:
# 할인율 구간 x Type별 지연률
# Type은 전처리 CSV에서는 숫자라 원본 train으로 보는 게 좋음

raw_train = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")

target = "Late_delivery_risk"
discount_col = "Order Item Discount Rate"

raw_train["discount_bin"] = pd.cut(
    raw_train[discount_col],
    bins=[-0.001, 0, 0.05, 0.10, 0.15, 0.20, 0.25],
    labels=["0%", "0-5%", "5-10%", "10-15%", "15-20%", "20-25%"]
)

pivot = raw_train.pivot_table(
    index="discount_bin",
    columns="Type",
    values=target,
    aggfunc="mean"
)

count_pivot = raw_train.pivot_table(
    index="discount_bin",
    columns="Type",
    values=target,
    aggfunc="count"
)

display(pivot)
display(count_pivot)

plt.figure(figsize=(9, 5))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".3f",
    cmap="coolwarm"
)

plt.title("Late Delivery Risk Rate: Discount Bin x Type")
plt.xlabel("Type")
plt.ylabel("Discount Bin")
plt.show()


In [ ]:
raw_train["price_bin"] = pd.qcut(
    raw_train["Product Price"],
    q=5,
    duplicates="drop"
)

pivot = raw_train.pivot_table(
    index="discount_bin",
    columns="price_bin",
    values=target,
    aggfunc="mean"
)

count_pivot = raw_train.pivot_table(
    index="discount_bin",
    columns="price_bin",
    values=target,
    aggfunc="count"
)

display(pivot)
display(count_pivot)

plt.figure(figsize=(12, 5))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".3f",
    cmap="coolwarm"
)

plt.title("Late Delivery Risk Rate: Discount Bin x Product Price Bin")
plt.xlabel("Product Price Bin")
plt.ylabel("Discount Bin")
plt.show()


In [ ]:
raw_train["order_dt"] = pd.to_datetime(raw_train["order date (DateOrders)"], errors="coerce")
raw_train["order_hour"] = raw_train["order_dt"].dt.hour

pivot = raw_train.pivot_table(
    index="discount_bin",
    columns="order_hour",
    values=target,
    aggfunc="mean"
)

count_pivot = raw_train.pivot_table(
    index="discount_bin",
    columns="order_hour",
    values=target,
    aggfunc="count"
)

display(pivot)
display(count_pivot)

plt.figure(figsize=(16, 5))
sns.heatmap(
    pivot,
    annot=False,
    cmap="coolwarm"
)

plt.title("Late Delivery Risk Rate: Discount Bin x Order Hour")
plt.xlabel("Order Hour")
plt.ylabel("Discount Bin")
plt.show()


In [ ]:
# 할인율 0% 주문 살펴보기

raw_train = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")

zero_discount = raw_train[raw_train["Order Item Discount Rate"] == 0].copy()

print("0% discount rows:", len(zero_discount))
print("ratio:", len(zero_discount) / len(raw_train))

display(zero_discount.head())


In [ ]:
# 할인율 0%인 주문만 describe().T

raw_train = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")

zero_discount = raw_train[raw_train["Order Item Discount Rate"] == 0].copy()

print("0% discount rows:", zero_discount.shape[0])
print("0% discount ratio:", zero_discount.shape[0] / raw_train.shape[0])

zero_desc = zero_discount.describe().T

zero_desc["missing"] = zero_discount.isnull().sum()
zero_desc["n_unique"] = zero_discount.nunique()
zero_desc["dtype"] = zero_discount.dtypes

zero_desc = zero_desc[
    ["dtype", "missing", "n_unique", "count", "mean", "std", "min", "25%", "50%", "75%", "max"]
]

display(zero_desc)


In [ ]:
raw_train = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")

target = "Late_delivery_risk"
discount_col = "Order Item Discount Rate"

raw_train["is_zero_discount"] = (raw_train[discount_col] == 0).astype(int)

compare_cols = [
    "Product Price",
    "Sales",
    "Order Item Total",
    "Order Item Discount",
    "Order Item Profit Ratio",
    "Order Profit Per Order",
    "Order Item Quantity",
    target
]

zero_vs_nonzero = (
    raw_train
    .groupby("is_zero_discount")[compare_cols]
    .agg(["count", "mean", "std", "min", "median", "max"])
)

display(zero_vs_nonzero)


In [ ]:
zero_discount = raw_train[raw_train["is_zero_discount"] == 1].copy()

num_cols = [
    "Product Price",
    "Sales",
    "Order Item Total",
    "Order Item Profit Ratio",
    "Order Profit Per Order",
    "Order Item Quantity"
]

outlier_rows = {}

outlier_summary = []

for col in num_cols:
    Q1 = zero_discount[col].quantile(0.25)
    Q3 = zero_discount[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    mask = (zero_discount[col] < lower) | (zero_discount[col] > upper)
    outlier_rows[col] = zero_discount[mask]
    
    outlier_summary.append({
        "feature": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "lower": lower,
        "upper": upper,
        "outlier_count": mask.sum(),
        "outlier_ratio": mask.mean(),
        "outlier_risk_rate": zero_discount.loc[mask, target].mean() if mask.sum() > 0 else np.nan,
        "normal_risk_rate": zero_discount.loc[~mask, target].mean()
    })

outlier_summary = pd.DataFrame(outlier_summary).sort_values("outlier_ratio", ascending=False)
display(outlier_summary)


In [ ]:
# 0% 할인군 카테고리별 count / risk rate

zero_cat = (
    zero_discount
    .groupby("Category Name")[target]
    .agg(["count", "mean"])
    .sort_values("count", ascending=False)
)

display(zero_cat.head(30))


In [ ]:
# Category Name 기준으로 의류(Apparel/Clothing)만 따로 묶기

raw_train = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")

target = "Late_delivery_risk"

apparel_categories = [
    "Women's Apparel",
    "Girls' Apparel",
    "Women's Clothing",
    "Men's Clothing",
    "Children's Clothing",
    "Baby",
    "Golf Apparel"
]

raw_train["is_clothing"] = raw_train["Category Name"].isin(apparel_categories).astype(int)

# 옷 vs 비옷 지연률 비교
clothing_summary = (
    raw_train
    .groupby("is_clothing")[target]
    .agg(["count", "mean"])
    .rename(columns={"mean": "late_risk_rate"})
)

display(clothing_summary)


In [ ]:
# 옷 여부 x 할인율 구간

discount_col = "Order Item Discount Rate"

raw_train["discount_bin"] = pd.cut(
    raw_train[discount_col],
    bins=[-0.001, 0, 0.05, 0.10, 0.15, 0.20, 0.25],
    labels=["0%", "0-5%", "5-10%", "10-15%", "15-20%", "20-25%"]
)

pivot = raw_train.pivot_table(
    index="discount_bin",
    columns="is_clothing",
    values=target,
    aggfunc="mean"
)

pivot.columns = ["non_clothing", "clothing"]

display(pivot)

plt.figure(figsize=(7, 5))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".3f",
    cmap="coolwarm"
)

plt.title("Risk Rate: Discount Bin x Clothing")
plt.xlabel("")
plt.ylabel("Discount Bin")
plt.show()


In [ ]:
# Category Name x Discount Bin별 Late_delivery_risk 비율/카운트

raw_train = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")

target = "Late_delivery_risk"
discount_col = "Order Item Discount Rate"

raw_train["discount_bin"] = pd.cut(
    raw_train[discount_col],
    bins=[-0.001, 0, 0.05, 0.10, 0.15, 0.20, 0.25],
    labels=["0%", "0-5%", "5-10%", "10-15%", "15-20%", "20-25%"]
)

# 카테고리별 전체 count 기준 정렬
category_order = (
    raw_train["Category Name"]
    .value_counts()
    .index
)

# 비율 pivot
category_discount_rate = raw_train.pivot_table(
    index="Category Name",
    columns="discount_bin",
    values=target,
    aggfunc="mean"
).reindex(category_order)

# count pivot
category_discount_count = raw_train.pivot_table(
    index="Category Name",
    columns="discount_bin",
    values=target,
    aggfunc="count"
).reindex(category_order)

display(category_discount_rate)
display(category_discount_count)


In [ ]:
# 원본 train에서 id 중복 확인

raw_train = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")

id_counts = raw_train["id"].value_counts()

print("전체 row 수:", len(raw_train))
print("고유 id 수:", raw_train["id"].nunique())
print("중복 id 개수:", (id_counts > 1).sum())

display(id_counts[id_counts > 1].head(30).to_frame("count"))


In [ ]:
raw_train = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")

pd.crosstab(raw_train["Type"], raw_train["Market"], normalize="index")


In [ ]:
for t in raw_train["Type"].unique():
    print(f"\n=== {t} ===")
    display(
        raw_train[raw_train["Type"] == t]["Category Name"]
        .value_counts(normalize=True)
        .head(10)
    )


In [ ]:
raw_train["order_dt"] = pd.to_datetime(raw_train["order date (DateOrders)"], errors="coerce")
raw_train["order_hour"] = raw_train["order_dt"].dt.hour

pd.crosstab(raw_train["Type"], raw_train["order_hour"], normalize="index")


In [ ]:
# Type별 count/risk
display(
    raw_train.groupby("Type")["Late_delivery_risk"]
    .agg(["count", "mean"])
    .sort_values("mean")
)


In [ ]:
# TRANSFER가 왜 Late_delivery_risk가 낮은지 확인용 요약 EDA

raw_train = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")

target = "Late_delivery_risk"

raw_train["order_dt"] = pd.to_datetime(
    raw_train["order date (DateOrders)"],
    errors="coerce"
)
raw_train["order_hour"] = raw_train["order_dt"].dt.hour

# 1. Type별 기본 지연률
print("=== 1. Type별 Late Delivery Risk ===")
type_summary = (
    raw_train
    .groupby("Type")[target]
    .agg(["count", "mean"])
    .rename(columns={"mean": "risk_rate"})
    .sort_values("risk_rate")
)
display(type_summary)

# 2. Type x Market 지연률
print("\n=== 2. Type x Market Risk Rate ===")
type_market_rate = raw_train.pivot_table(
    index="Type",
    columns="Market",
    values=target,
    aggfunc="mean"
)

type_market_count = raw_train.pivot_table(
    index="Type",
    columns="Market",
    values=target,
    aggfunc="count"
)

display(type_market_rate)
display(type_market_count)

plt.figure(figsize=(8, 4))
sns.heatmap(
    type_market_rate,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    center=raw_train[target].mean()
)
plt.title("Risk Rate: Type x Market")
plt.show()

# 3. Type x Order Region 지연률: 상위 region만
print("\n=== 3. Type x Top Order Region Risk Rate ===")
top_regions = raw_train["Order Region"].value_counts().head(10).index

region_df = raw_train[raw_train["Order Region"].isin(top_regions)]

type_region_rate = region_df.pivot_table(
    index="Type",
    columns="Order Region",
    values=target,
    aggfunc="mean"
)

type_region_count = region_df.pivot_table(
    index="Type",
    columns="Order Region",
    values=target,
    aggfunc="count"
)

display(type_region_rate)
display(type_region_count)

plt.figure(figsize=(14, 4))
sns.heatmap(
    type_region_rate,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    center=raw_train[target].mean()
)
plt.title("Risk Rate: Type x Top Order Region")
plt.show()

# 4. Type x Order Hour 지연률
print("\n=== 4. Type x Order Hour Risk Rate ===")
type_hour_rate = raw_train.pivot_table(
    index="Type",
    columns="order_hour",
    values=target,
    aggfunc="mean"
)

type_hour_count = raw_train.pivot_table(
    index="Type",
    columns="order_hour",
    values=target,
    aggfunc="count"
)

display(type_hour_rate)
display(type_hour_count)

plt.figure(figsize=(16, 4))
sns.heatmap(
    type_hour_rate,
    annot=False,
    cmap="coolwarm",
    center=raw_train[target].mean()
)
plt.title("Risk Rate: Type x Order Hour")
plt.show()

# 5. Type별 카테고리 구성
print("\n=== 5. Type별 Top Category 구성 ===")
for t in type_summary.index:
    print(f"\n--- {t} ---")
    display(
        raw_train[raw_train["Type"] == t]["Category Name"]
        .value_counts(normalize=True)
        .head(10)
        .to_frame("category_ratio")
    )

# 6. Type별 수치형 평균 비교
print("\n=== 6. Type별 주요 수치형 평균 ===")
num_compare_cols = [
    "Order Item Discount Rate",
    "Order Item Discount",
    "Order Item Quantity",
    "Sales",
    "Order Item Total",
    "Order Profit Per Order",
    "Product Price"
]

display(
    raw_train
    .groupby("Type")[num_compare_cols]
    .mean()
    .round(3)
)
